# YouTube 4K 人像驗證 (Stage 2 Tensor / Colab T4)

本筆記本依據 `4K人像測試_PLAN.md` 規格執行：
- 執行環境：Google Colab T4 GPU (Python 3.13+, PyTorch 2.11+, TorchCodec 0.17.0 cu130)
- 資料流：固定全畫面 Stage 2 Tensor 推論，無 precrop，輸出 1080x1920 HEVC (NVENC)
- 追蹤排程：支援 `balanced` 與 `conservative`
- 驗證原則：每一 clip/profile 先執行 180 幀短測，通過後再執行完整 30-60 秒輸出
- 預算限制：T4 累積連線時間至多 120 分鐘；結果同步備份至 Google Drive

In [ ]:
import sys, subprocess, json
from pathlib import Path

def checked_capture(command):
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.returncode:
        print(completed.stdout, end="")
        print(completed.stderr, end="", file=sys.stderr)
        completed.check_returncode()
    return completed

if sys.version_info < (3, 13):
    raise RuntimeError("需要 Python 3.13+，請選擇相容的 Colab runtime")

check = checked_capture([sys.executable, "-c", """
import json, torch, torchvision
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.11"), torch.__version__
assert torch.cuda.is_available(), "請開啟 GPU runtime；不允許 CPU fallback"
gpu_name = torch.cuda.get_device_name(0)
assert "T4" in gpu_name, f"必須為 T4 GPU；偵測到：{gpu_name}"
print(json.dumps({"torch": torch.__version__, "torchvision": torchvision.__version__,
                  "cuda_build": torch.version.cuda, "gpu": gpu_name}))
"""])
accelerators = json.loads(check.stdout)
print("GPU 環境確認：", accelerators)

### 1. 套件安裝與驗證
僅在環境尚未安裝相容 TorchCodec 時執行。固定安裝 TorchCodec 0.17.0 cu130，並驗證 torchvision CUDA NMS 與 FFmpeg/NVENC。

In [ ]:
repo = Path("/content/auto-vertical-reframe")
# 若為上傳快照則解壓縮至 repo
snapshot_tar = Path("/content/reframe-snapshot.tar.gz")
if snapshot_tar.is_file() and not repo.exists():
    subprocess.run(["tar", "-xzf", str(snapshot_tar), "-C", "/content"], check=True)

if repo.exists():
    subprocess.run([sys.executable, str(repo / "install_colab.py")], cwd=repo, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall",
                "torchcodec==0.17.0", "--index-url", "https://download.pytorch.org/whl/cu130"],
               check=True)

verify = checked_capture([sys.executable, "-c", """
import json, shutil, torch, torchvision, torchcodec
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "需要 FFmpeg / ffprobe CLI"
from torchcodec.decoders import VideoDecoder
boxes = torch.tensor([[0., 0., 2., 2.]], device="cuda")
scores = torch.tensor([1.], device="cuda")
assert torchvision.ops.nms(boxes, scores, 0.5).numel() == 1, "torchvision CUDA NMS 驗證失敗"
print(json.dumps({"torch": torch.__version__, "torchvision": torchvision.__version__,
                  "torchcodec": torchcodec.__version__}))
"""])
print("套件驗證完成：", verify.stdout)

### 2. 掛載 Google Drive 與工作目錄配置

In [ ]:
from google.colab import drive
import os, shutil, sys, json
from pathlib import Path

drive.mount("/content/drive")
repo = Path("/content/auto-vertical-reframe")
DRIVE_ROOT = Path("/content/drive/MyDrive/reframe-youtube-validation")
RUN_DATE = "20261013"  # 測試日期標記
BACKUP_DIR = DRIVE_ROOT / f"{RUN_DATE}-runs"
LOCAL_OUT = Path("/content/reframe-youtube-validation") / f"{RUN_DATE}-runs"
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

SEG_MODEL = Path("/content/yolo26n-seg.pt")
POSE_MODEL = Path("/content/yolo26n-pose.pt")
os.environ["TORCH_HOME"] = str(DRIVE_ROOT / "torch-cache")
print("本機工作目錄：", LOCAL_OUT)
print("Drive 備份目錄：", BACKUP_DIR)

### 3. 執行短測 (180 幀) 與完整片段測試
每一片段先以 `--max-frames 180` 驗證，確認硬體解碼無 fallback 且尺寸與時戳正確後，再執行全片。

In [ ]:
CLIP_PATH = Path("/content/sample-clip.mp4")  # 指定受測 4K CFR 片段
TRACKING_PROFILE = "balanced"                 # balanced 或 conservative

# 執行 180 幀短測
cmd_short = [
    sys.executable, "-u", str(repo / "scripts/youtube_validation.py"),
    "--input", str(CLIP_PATH),
    "--output-dir", str(LOCAL_OUT),
    "--backup-dir", str(BACKUP_DIR),
    "--seg-model", str(SEG_MODEL),
    "--pose-model", str(POSE_MODEL),
    "--tracking-profile", TRACKING_PROFILE,
    "--max-frames", "180"
]
subprocess.run(cmd_short, cwd=repo, check=True)

# 短測通過後，執行全片段
RUN_FULL = False
if RUN_FULL:
    cmd_full = [
        sys.executable, "-u", str(repo / "scripts/youtube_validation.py"),
        "--input", str(CLIP_PATH),
        "--output-dir", str(LOCAL_OUT),
        "--backup-dir", str(BACKUP_DIR),
        "--seg-model", str(SEG_MODEL),
        "--pose-model", str(POSE_MODEL),
        "--tracking-profile", TRACKING_PROFILE,
    ]
    subprocess.run(cmd_full, cwd=repo, check=True)

### 4. 檢視結果與日誌指標

In [ ]:
for rec_path in sorted(LOCAL_OUT.glob("*/record.json")):
    rec = json.loads(rec_path.read_text(encoding="utf-8"))
    print(f"目錄: {rec_path.parent.name}")
    print(f"  狀態: {rec.get('status')}")
    print(f"  Profile: {rec.get('tracking_profile')}")
    print(f"  耗時: {rec.get('subprocess_wall_seconds')}s")
    if 'summary' in rec:
        s = rec['summary']
        print(f"  處理幀數: {s.get('frames_processed')}")
        print(f"  分割呼叫數: {s.get('seg_detector_calls')}")
        print(f"  Pose 快取命中: {s.get('pose_extension_hits')}")
    print("-" * 50)